# MLP & DNN Regression — Jena Climate Dataset
**Target:** `T (degC)` · **Input:** `output/processed_data.csv` from Stage 1

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from utils.data_prep import load_and_split, scale_features
from utils.models import build_mlp, build_dnn
from utils.trainer import compile_and_fit
from utils.evaluator import compute_metrics, metrics_table, plot_loss_curves, plot_actual_vs_predicted

DATA_PATH   = Path('../output/processed_data.csv')
SCALER_PATH = Path('../output/scaler.pkl')
MLP_PATH    = Path('../output/mlp_best.keras')
DNN_PATH    = Path('../output/dnn_best.keras')

print('TF version:', tf.__version__)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

---
## Section 1 — Data Preparation

In [ ]:
train_df, val_df, test_df = load_and_split(DATA_PATH)

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, scaler = scale_features(
    train_df, val_df, test_df, scaler_path=SCALER_PATH
)
INPUT_DIM = X_train.shape[1]
print(f'Input dimension: {INPUT_DIM}')

---
## Section 2 — MLP Model

In [ ]:
mlp = build_mlp(INPUT_DIM)
mlp.summary()

In [ ]:
mlp_history = compile_and_fit(
    mlp, X_train, y_train, X_val, y_val,
    checkpoint_path=MLP_PATH,
    epochs=200,
    batch_size=256,
    patience=10,
)

---
## Section 3 — DNN Model

In [ ]:
dnn = build_dnn(INPUT_DIM)
dnn.summary()

In [ ]:
dnn_history = compile_and_fit(
    dnn, X_train, y_train, X_val, y_val,
    checkpoint_path=DNN_PATH,
    epochs=200,
    batch_size=256,
    patience=10,
)

In [ ]:
# Overlay training curves: MLP vs DNN
plot_loss_curves({"MLP": mlp_history, "DNN": dnn_history})

---
## Section 4 — Evaluation

In [ ]:
splits = {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}

mlp_metrics = compute_metrics(mlp, splits)
dnn_metrics = compute_metrics(dnn, splits)

table = metrics_table(mlp_metrics, dnn_metrics)
print(table.to_string())
table

In [ ]:
# Pick the winner by test RMSE
mlp_test_rmse = mlp_metrics['test']['RMSE']
dnn_test_rmse = dnn_metrics['test']['RMSE']
winner_name = 'MLP' if mlp_test_rmse <= dnn_test_rmse else 'DNN'
winner_model = mlp if winner_name == 'MLP' else dnn
print(f'Winner: {winner_name}  (MLP test RMSE={mlp_test_rmse:.4f} | DNN test RMSE={dnn_test_rmse:.4f})')

plot_actual_vs_predicted(winner_model, X_test, y_test, model_name=winner_name, n=500)

### Conclusion

Both models exploit the lag and rolling features engineered in Stage 1, so even a shallow MLP captures the autocorrelation structure well.

**MLP (3 hidden layers):** Fast convergence, low variance. With only 3 layers the network cannot overfit easily, and EarlyStopping stops training quickly.

**DNN (6 hidden layers + Dropout):** Larger capacity, but Dropout regularisation is needed to match or beat the MLP on val/test. For a tabular regression task where the predictive signal is already distilled into lag/rolling features, the extra depth provides diminishing returns — the MLP tends to be competitive or slightly better.

**Why depth alone does not help here:** The lag and rolling features make the regression nearly linear in the recent temperature history. A wider, shallower network can fit this as well as a very deep one, with less risk of over-regularising via Dropout.

---
## Section 5 — Save & Verify Artifacts

In [ ]:
# Confirm checkpoint files exist
for p in [MLP_PATH, DNN_PATH, SCALER_PATH]:
    size = p.stat().st_size if p.exists() else -1
    print(f'{p.name}: {"OK " + str(size) + " bytes" if size >= 0 else "MISSING"}')

In [ ]:
# Reload MLP from disk and run one forward pass to verify
import tensorflow as tf

mlp_reloaded = tf.keras.models.load_model(MLP_PATH)
sample_pred = mlp_reloaded.predict(X_test[:5], verbose=0).flatten()
print('Reloaded MLP predictions on first 5 test samples (°C):')
print(sample_pred)
print('Ground truth:')
print(y_test[:5])